In [1]:
from pathlib import Path
import sys
import numpy as np
def ensure_tests_on_path(max_up=10):
    p = Path.cwd().resolve()
    for _ in range(max_up):
        if (p / 'tests').is_dir():
            sys.path.insert(0, str(p))
            return str(p)
        if p.parent == p:
            break
        p = p.parent
    # fallback: insert cwd
    sys.path.insert(0, str(Path.cwd().resolve()))
    return str(Path.cwd().resolve())

root_added = ensure_tests_on_path()
print("Added to sys.path:", root_added)


Added to sys.path: /Users/ashmi/Code/Scripts/phd/travel-crs


In [2]:
DIMENSIONS = ["relevance", "sustainability", "popularity_balance", "diversity"]

In [12]:
import json
from tests.analysis.src.quantitative import *
from tests.analysis.src.utils import load_listwise_evaluations_df, add_numeric_scores

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

/Users/ashmi/Code/Scripts/phd/travel-crs/.crs-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
%load_ext autoreload
%autoreload 2

In [14]:
def avg_pairwise_explanation_similarity_across_queries(
    dfs_by_model,
    dimensions,
    explanation_suffix="_explanation",
    sbert_model="all-MiniLM-L6-v2",
    device=None,
):
    """
    Return a dict mapping each dimension -> DataFrame (models x models) containing
    the average cosine similarity across queries for each model pair.
    - dfs_by_model: dict like {'gpt': df_gpt, 'gemini': df_gemini, 'deepseek': df_deepseek}
    - dimensions: list of base dimension names, e.g. ['relevance', ...]
    - Only queries where both models have a non-empty explanation are used for that pair.
    """
    embedder = SentenceTransformer(sbert_model, device=device) if device else SentenceTransformer(sbert_model)
    models = list(dfs_by_model.keys())

    # collect all query ids present in any dataframe
    all_queries = set()
    for df in dfs_by_model.values():
        if "query_id" in df.columns:
            all_queries.update(df["query_id"].dropna().unique())
    all_queries = sorted(all_queries)

    # accumulators: accum[dim][i][j] -> list of similarity scores across queries
    accum = {dim: [[[] for _ in models] for _ in models] for dim in dimensions}

    for qid in all_queries:
        for dim in dimensions:
            col = f"{dim}{explanation_suffix}"
            texts = []
            present = []
            for m in models:
                df = dfs_by_model[m]
                if col in df.columns:
                    rows = df.loc[df["query_id"] == qid, col]
                    val = rows.iloc[0] if len(rows) > 0 else None
                    if pd.isna(val) or (isinstance(val, str) and val.strip() == ""):
                        val = None
                else:
                    val = None
                present.append(val is not None)
                texts.append(val or "")  # placeholder for encoder

            # embed all (placeholder empty strings will be ignored via 'present' mask)
            embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
            sim = cosine_similarity(embeddings)

            # accumulate pairwise similarities only when both present
            n = len(models)
            for i in range(n):
                for j in range(n):
                    if present[i] and present[j]:
                        accum[dim][i][j].append(float(sim[i, j]))

    # Build averaged DataFrames per dimension
    avg_by_dimension = {}
    for dim in dimensions:
        n = len(models)
        mat = np.full((n, n), np.nan, dtype=float)
        for i in range(n):
            for j in range(n):
                vals = accum[dim][i][j]
                if vals:
                    mat[i, j] = float(np.mean(vals))
        avg_by_dimension[dim] = pd.DataFrame(mat, index=models, columns=models)

    return avg_by_dimension

In [21]:
version = "v1"
def run_exp(version):
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gemini_eval_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gpt5_eval_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/deepseek_eval_{version}.json'
    
    selected_cols = ["query_id", "judge_model", \
                "relevance_explanation", "relevance_confidence", \
               "diversity_explanation", "sustainability_explanation", "sustainability_confidence", \
               "popularity_balance_explanation", "popularity_balance_confidence"]
        
    df_gpt = load_listwise_evaluations_df(
        gpt_eval_path, model_name="gpt5", version=version)[selected_cols]
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)[selected_cols]
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)[selected_cols]
    dfs_by_model = {'gpt': df_gpt, 'gemini': df_gemini, 'deepseek': df_deepseek}

    res = avg_pairwise_explanation_similarity_across_queries(
    dfs_by_model,
    DIMENSIONS,
    explanation_suffix="_explanation",
    sbert_model="all-MiniLM-L6-v2",
    device=None,
    )
    return res


In [22]:
res = run_exp("v1")
res

ℹ️ Loaded 101 successful evaluation entries for gpt5, version v1.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v1.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v1.


{'relevance':                gpt    gemini  deepseek
 gpt       1.000000  0.720689  0.645682
 gemini    0.720689  1.000000  0.632919
 deepseek  0.645682  0.632919  1.000000,
 'diversity':                gpt    gemini  deepseek
 gpt       1.000000  0.737238  0.696205
 gemini    0.737238  1.000000  0.751508
 deepseek  0.696205  0.751508  1.000000,
 'popularity_balance':                gpt    gemini  deepseek
 gpt       1.000000  0.723224  0.722718
 gemini    0.723224  1.000000  0.758375
 deepseek  0.722718  0.758375  1.000000,
 'sustainability':                gpt    gemini  deepseek
 gpt       1.000000  0.733025  0.642273
 gemini    0.733025  1.000000  0.659866
 deepseek  0.642273  0.659866  1.000000}

In [23]:
res = run_exp("v2")
res

ℹ️ Loaded 101 successful evaluation entries for gpt5, version v2.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v2.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v2.


{'relevance':                gpt    gemini  deepseek
 gpt       1.000000  0.682752  0.622516
 gemini    0.682752  1.000000  0.641001
 deepseek  0.622516  0.641001  1.000000,
 'diversity':                gpt    gemini  deepseek
 gpt       1.000000  0.784046  0.730229
 gemini    0.784046  1.000000  0.752324
 deepseek  0.730229  0.752324  1.000000,
 'popularity_balance':                gpt    gemini  deepseek
 gpt       1.000000  0.721669  0.679956
 gemini    0.721669  1.000000  0.744738
 deepseek  0.679956  0.744738  1.000000,
 'sustainability':                gpt    gemini  deepseek
 gpt       1.000000  0.740235  0.662039
 gemini    0.740235  1.000000  0.694961
 deepseek  0.662039  0.694961  1.000000}